In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from gdt.core.data_primitives import TimeBins
from bctools.analysis import BayesianBlocksLightcurve
import os

grb_name = "bn240810880"
base_path = "/home/cosi/cosi/data/grb_eliza/test_t90"
data = np.load(base_path+"/"+grb_name+"light_curve.npz")
total_lc = data['light_curve']

from gdt.core.background.binned import Polynomial


In [ ]:



def analyze_lc(lightcurve, p0=0.05, isRate=False, panels=['z0', 'z1', 'x0', 'x1', 'y0', 'y1']):
    """
    Analyze the light curve data.
    Input:
        - lightcurve: data file with times and counts
        - p0: false alarm probability for the Bayesian Block algorithm
        - isRate: if True, the lightcurve files contains rates instead of counts
        - panels: list of detectors for which we have lightcurves
    """

    data = lightcurve  
    time = data[:,0,0] 
    #time = time - time[0]

    signal = {}
    for i, panel in enumerate(panels):
        signal[panel] = data[:, i, 1]
        if isRate:
            signal[panel] = signal[panel] * bin_width

    
    # Construct light curve object
    bin_width = time[1] - time[0]
    bin_edges = np.zeros(len(time) + 1)
    bin_edges[1:-1] = (time[:-1] + time[1:]) / 2
    bin_edges[0] = time[0] - (time[1] - time[0]) / 2
    bin_edges[-1] = time[-1] + (time[-1] - time[-2]) / 2
    lo_edges, hi_edges = bin_edges[:-1], bin_edges[1:]
    exposure = np.full(len(time), bin_width)


    lc = {} # lc per panel
    for panel in panels:
        signal_panel = signal[panel]
        lc[panel] = TimeBins(signal_panel, lo_edges, hi_edges, exposure)
    #if len(panels) > 1: lc_psum = TimeBins.sum([lcs for lcs in lc.values()])
    #else: lc_psum = lc[panels[0]]

    
    best_panel = None
    best_value = float('-inf')

    for i in range(6):
        panel = panels[i]
        value = max(lc[panel].counts)

        if value > best_value:
            best_value = value
            best_panel = panel
            
    lc_psum = lc[best_panel]

    lc_sel = lc_psum
   

    # Apply Bayesian Blocks algorithm

    try: 
        bb_lc = BayesianBlocksLightcurve(lc_sel)
        
        bb_lc.compute_bayesian_blocks(p0=p0)
        
        signal_range = bb_lc.signal_range
        
        t90 = bb_lc.duration(quantile = .9)
        
        t90_error = bb_lc.duration_error(.9, nsamples = 100)
        
    except Exception as e:
        print(e)
        print('WARNING: ')
        return lc_sel, None, -9999, -9999, -9999, -9999, -9999, -9999, -9999, None, None
   

    # Li&Ma calculation of the significance
    signal_lc = lc_sel.slice(bb_lc.signal_range.tstart, bb_lc.signal_range.tstop)
    bkg_lc = lc_sel.slice(bb_lc.signal_range.tstop, lc_sel.centroids[-1])
    bkg_lc = lc_sel.slice(lc_sel.centroids[0], bb_lc.signal_range.tstart)
    t_on = np.sum(signal_lc.exposure)
    t_off = np.sum(bkg_lc.exposure)
    alpha = t_on / t_off
    N_on = np.sum(signal_lc.rates * signal_lc.exposure)
    N_off = np.sum(bkg_lc.rates * bkg_lc.exposure)
    S = np.sqrt(2) * ( N_on * np.log( ((1+alpha)/alpha) * (N_on/(N_on+N_off)) ) + N_off * np.log( (1+alpha) * (N_off/(N_on+N_off)) ) ) ** 0.5

    # Li&Ma calculation of the peak significance
    significance = []
    for rate, exp in zip(signal_lc.rates, signal_lc.exposure):
        N_on = rate * exp
        t_on = exp
        alpha = t_on / t_off
        S_bin = np.sqrt(2) * ( N_on * np.log( ((1+alpha)/alpha) * (N_on/(N_on+N_off)) ) + N_off * np.log( (1+alpha) * (N_off/(N_on+N_off)) ) ) ** 0.5
        significance.append(S_bin)
    significance = np.array(significance)
    S_peak = np.max(significance)

    result = (lc_sel, bb_lc, lc, signal_range.tstart, signal_range.tstop, t90, t90_error[0], t90_error[1], S, S_peak, lo_edges,hi_edges)
    plt.show
 
    return result


In [ ]:
def fit_background_gdt(lc, signal_range, buffer=0.0, order=2):
    """
    Fit del background polinomiale su una light curve GDT (TimeBins),
    escludendo la finestra del segnale.

    Parameters
    ----------
    lc : TimeBins
        Light curve del detector.
        Deve avere almeno: counts, lo_edges, hi_edges, exposure
    signal_range : tuple
        (tstart, tstop) del segnale/burst da escludere dal fit
    buffer : float, optional
        Margine extra da escludere attorno al segnale
    order : int, optional
        Ordine del polinomio

    Returns
    -------
    result : dict
        Dizionario con:
        - "model"           : oggetto Polynomial fittato
        - "mask_bkg"        : maschera booleana dei bin usati nel fit
        - "bkg_rate"        : background stimato in rate
        - "bkg_rate_err"    : errore sul background rate
        - "bkg_counts"      : background stimato in counts/bin
        - "bkg_counts_err"  : errore in counts/bin
        - "net_counts"      : counts osservati - background counts
        - "net_rate"        : rate osservato - background rate
    """

    tstart_sig = signal_range.tstart
    tstop_sig = signal_range.tstop
    excl_start = tstart_sig - buffer
    excl_stop = tstop_sig + buffer

    # bin completamente fuori dalla regione esclusa
    mask_bkg = (lc.hi_edges <= excl_start) | (lc.lo_edges >= excl_stop)

    n_bkg_bins = np.sum(mask_bkg)
    if n_bkg_bins < (order + 2):
        raise RuntimeError(
            f"Troppi pochi bin di background ({n_bkg_bins}) "
            f"per un polinomio di ordine {order}"
        )

    # costruiamo il modello come in bctools
    bkg_model = Polynomial(
        counts=lc.counts[mask_bkg][:, np.newaxis],
        tstart=lc.lo_edges[mask_bkg],
        tstop=lc.hi_edges[mask_bkg],
        exposure=lc.exposure[mask_bkg]
    )

    bkg_model.fit(order=order)

    # ATTENZIONE: interpolate() restituisce RATE, non counts
    bkg_rate, bkg_rate_err = bkg_model.interpolate(
        tstart=lc.lo_edges,
        tstop=lc.hi_edges
    )

    # da shape (N, 1) a (N,)
    bkg_rate = np.squeeze(bkg_rate)
    bkg_rate_err = np.squeeze(bkg_rate_err)

    # conversione a counts/bin
    bkg_counts = bkg_rate * lc.exposure
    bkg_counts_err = bkg_rate_err * lc.exposure

    # osservati
    obs_rate = lc.counts / lc.exposure

    # netti
    net_counts = lc.counts - bkg_counts
    net_rate = obs_rate - bkg_rate

    return {
        "model": bkg_model,
        "mask_bkg": mask_bkg,
        "bkg_rate": bkg_rate,
        "bkg_rate_err": bkg_rate_err,
        "bkg_counts": bkg_counts,
        "bkg_counts_err": bkg_counts_err,
        "net_counts": net_counts,
        "net_rate": net_rate,
    }

In [ ]:
total_lc.shape

In [ ]:
output = analyze_lc(total_lc, p0=10e-5, isRate=False, panels=['z1', 'z0', 'x1', 'x0', 'y1', 'y0'])



In [ ]:
bb_lc = output[1]
lc_sel = output[0]
signal_range = bb_lc.signal_range

fig = plt.figure(figsize=(10,4))


plt.step(lc_sel.centroids, lc_sel.counts, where="mid")

plt.xlabel("Time [s]")
plt.ylabel("Counts / bin")
plt.title("Light curve")
plt.grid(True, alpha=0.3)

plt.show()

fig = plt.figure(figsize=(10,4))


plt.plot(lc_sel.centroids, bb_lc.bkg_counts/lc_sel.exposure, color = 'red', ls = ':',
                label = "Fitted background")
plt.errorbar(lc_sel.centroids, lc_sel.rates, xerr = [lc_sel.centroids-lc_sel.lo_edges, 
 lc_sel.hi_edges-lc_sel.centroids],
                    yerr = lc_sel.rate_uncertainty, 
                    ls = 'none', color = '.7',
                    label = 'Raw data')

lc_bayes = bb_lc.bb_lightcurve

plt.plot(np.append(lc_bayes.lo_edges, lc_bayes.hi_edges[-1]),
                np.append(lc_bayes.rates, lc_bayes.rates[-1]),
                drawstyle = 'steps-post',
                label = 'Bayesian blocks')
        
# Vertical lines showing the start and stop of the identified signal
plt.axvline(bb_lc.signal_range.tstart, ls = "--", color = 'olive', label = "Signal start/stop")
plt.axvline(bb_lc.signal_range.tstop, ls = "--", color = 'olive')

plt.legend()

In [ ]:
lc_array = output[2]

In [ ]:
signal_range.tstart

In [ ]:
signal_range.tstop-signal_range.tstart

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

panels = ['z1','z0','x1','x0','y1','y0']

results = []
signal_counts_arr = []
background_counts_arr = []
net_counts_arr = []

tstart = signal_range.tstart
tstop = signal_range.tstop
duration = tstop-tstart
for p in panels:
    
    print("Panel:", p)
   
    lc = lc_array[p]
    res = fit_background_gdt(lc, signal_range, buffer=1.0, order=2)

    results.append(res)
    
    # maschera della finestra del segnale
    mask_sig = (lc.lo_edges >= tstart) & (lc.hi_edges <= tstop)

    # somme nella finestra [tstart, tstop]
    signal_counts = np.sum(lc.counts[mask_sig])
    background_counts = np.sum(res["bkg_counts"][mask_sig])
    net_counts = signal_counts - background_counts

    signal_counts_arr.append(signal_counts)
    background_counts_arr.append(background_counts)
    net_counts_arr.append(net_counts)
    
    # plot
    t = 0.5 * (lc.lo_edges + lc.hi_edges)
    obs_rate = lc.counts / lc.exposure

    plt.figure(figsize=(10, 5))

    # counts osservati
    plt.step(t, lc.counts, where="mid", label="Observed counts")

    # background in counts
    plt.plot(t, res["bkg_counts"], label="Background (counts)")

    plt.axvspan(signal_range.tstart, signal_range.tstop, alpha=0.2, label="Signal window")

    plt.xlabel("Time")
    plt.ylabel("Counts / bin")
    

    plt.title(
        f"{p} | signal={signal_counts:.1f}, "
        f"bkg={background_counts:.1f}, net={net_counts:.1f}"
    )

    plt.legend()
    plt.show()
# conversione finale in array numpy
signal_counts_arr = np.array(signal_counts_arr)
background_counts_arr = np.array(background_counts_arr)
net_counts_arr = np.array(net_counts_arr)

print("Signal counts per panel:", signal_counts_arr)
print("Background counts per panel:", background_counts_arr)
print("Net counts per panel:", net_counts_arr)

In [ ]:
for p, s, b, n in zip(panels, signal_counts_arr, background_counts_arr, net_counts_arr):
    print(f"{p}: signal={s:.2f}, background={b:.2f}, net={n:.2f}")

In [ ]:
background_counts_arr

In [ ]:
signal_counts_arr

# Significance Analysis

In [ ]:
import math
def li_ma(s,b,t_on,t_off):
    
    alpha = t_on/t_off
    
    SA = math.sqrt(2) * math.sqrt(s * math.log(((1 + alpha) / alpha) * ( s / (s + b))) + b * math.log((1 + alpha) * (b / ( s + b ))));
    
    return SA


In [ ]:
print(duration)

In [ ]:
signal_counts_arr = np.array(signal_counts_arr)
background_counts_arr = np.array(background_counts_arr)
net_counts_arr = np.array(net_counts_arr)

sigmas = []

for S, B in zip(signal_counts_arr, background_counts_arr):
    if S < 10 or B < 10:
        sigmas.append(0)
    else:
        sigmas.append(li_ma(S,B,duration,duration))
        
sigmas = np.array(sigmas)

In [ ]:
print("Max sigma: "+str(np.max(sigmas)))

In [ ]:
sigma_tot = np.sqrt(np.sum(sigmas**2))
print("Quadrature sigma: "+str(sigma_tot))